In [21]:
import pandas as pd
import numpy as np
import joblib

import dice_ml
from dice_ml import Dice

In [22]:
xgb_model = joblib.load("xgb_model.pkl")
features = joblib.load("features.pkl")

print("Model loaded ✅")
print("Features:", features)

Model loaded ✅
Features: ['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest', 'newbalanceDest', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']


In [ ]:
df = pd.read_csv("fraud_sample.csv")

# Drop unwanted columns
df = df.drop(['nameOrig', 'nameDest', 'step', 'isFlaggedFraud'], axis=1, errors='ignore')

# One-hot encode
df = pd.get_dummies(df, columns=['type'], drop_first=True)

# 🔥 FORCE EVERYTHING TO FLOAT (CRITICAL FIX)
df = df.astype(float)

# Align with model
df = df.reindex(columns=features + ['isFraud'], fill_value=0)

# Fill any NaNs
df = df.fillna(0)

print(df.dtypes)

amount            float64
oldbalanceOrg     float64
newbalanceOrig    float64
oldbalanceDest    float64
newbalanceDest    float64
type_CASH_OUT     float64
type_DEBIT        float64
type_PAYMENT      float64
type_TRANSFER     float64
isFraud           float64
dtype: object


In [24]:
data_dice = dice_ml.Data(
    dataframe=df,
    continuous_features=features,   # 🔥 ALL FEATURES
    outcome_name='isFraud'
)

In [25]:
def safe_get_decimal_precisions(self, output_type="list"):
    return [2] * len(self.data_df.columns)

data_dice.get_decimal_precisions = safe_get_decimal_precisions.__get__(data_dice)

print("Precision fix applied ✅")

Precision fix applied ✅


In [26]:
model_dice = dice_ml.Model(
    model=xgb_model,
    backend="sklearn"
)

dice = Dice(data_dice, model_dice, method="genetic")

print("Dice ready 🚀")

Dice ready 🚀


In [27]:
fraud_sample = df[df['isFraud'] == 1].iloc[0:1]

fraud_sample

,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,isFraud
2,181.0,181.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0


In [28]:
query_instance = fraud_sample.drop('isFraud', axis=1).copy()

# Align strictly
query_instance = query_instance.reindex(columns=features, fill_value=0)

# Convert to numeric
query_instance = query_instance.astype(float)

query_instance.reset_index(drop=True, inplace=True)

print("Query ready ✅")

Query ready ✅


In [29]:
print("Query columns:", query_instance.columns)
print("Invalid types:", query_instance.select_dtypes(include=['object']).columns)

Query columns: Index(['amount', 'oldbalanceOrg', 'newbalanceOrig', 'oldbalanceDest',
       'newbalanceDest', 'type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT',
       'type_TRANSFER'],
      dtype='str')
Invalid types: Index([], dtype='str')


In [30]:
cf = dice.generate_counterfactuals(
    query_instance,
    total_CFs=3,
    desired_class="opposite"
)

100%|██████████| 1/1 [00:24<00:00, 24.52s/it]


In [31]:
cf.cf_examples_list[0].final_cfs_df

,amount,oldbalanceOrg,newbalanceOrig,oldbalanceDest,newbalanceDest,type_CASH_OUT,type_DEBIT,type_PAYMENT,type_TRANSFER,isFraud
0,0.00,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
0,162.17,174.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
0,0.00,171.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0
